# Data Preprocessing - VN30 Stock Data

Notebook này chuyển kết quả audit từ `01_eda.ipynb` thành các rule làm sạch nhất quán để tạo clean panel phục vụ tính return, correlation, clustering và optimization.

Trạng thái raw data theo EDA hiện tại:
- raw data khá sạch
- không duplicate
- không missing
- không lỗi parse

Các vấn đề cần được xử lý rõ bằng preprocess:
- coverage theo ngày chưa đều
- có `1` dòng `volume == 0`
- có `2` dòng vi phạm logic OHLC cần loại
- `SSB` có history ngắn hơn đáng kể do IPO muộn, nên không phù hợp với panel nghiên cứu cân bằng hiện tại


## Bước 0: Load & Chuẩn hóa cơ bản


In [1]:
from pathlib import Path
import pandas as pd
import numpy as np
import json

# Setup paths
PROJECT_ROOT = Path.cwd().parent
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS_DIR = PROJECT_ROOT / "reports" / "preprocess"

# Create directories
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"✓ Project root: {PROJECT_ROOT}")
print(f"✓ Raw data: {DATA_RAW}")
print(f"✓ Processed data: {DATA_PROCESSED}")
print(f"✓ Reports: {REPORTS_DIR}")


✓ Project root: d:\Archive\NCKH\CODE\vn30-ver1
✓ Raw data: d:\Archive\NCKH\CODE\vn30-ver1\data\raw
✓ Processed data: d:\Archive\NCKH\CODE\vn30-ver1\data\processed
✓ Reports: d:\Archive\NCKH\CODE\vn30-ver1\reports\preprocess


In [2]:
# Load raw data
df_raw = pd.read_csv(DATA_RAW / "vn30_stock_data.csv")

print(f"Loaded {len(df_raw):,} rows")
print(f"Columns: {list(df_raw.columns)}")
print(f"\nFirst rows:")
df_raw.head()


Loaded 43,932 rows
Columns: ['date', 'open', 'high', 'low', 'close', 'volume', 'symbol']

First rows:


,date,open,high,low,close,volume,symbol
0,2020-01-02,6.58,6.64,6.55,6.64,1163109,ACB
1,2020-01-03,6.64,6.70,6.61,6.64,1055528,ACB
2,2020-01-06,6.64,6.64,6.49,6.49,1286035,ACB
3,2020-01-07,6.49,6.55,6.49,6.49,1050934,ACB
4,2020-01-08,6.49,6.49,6.35,6.38,2304937,ACB


In [3]:
# Normalize raw data
df = df_raw.copy()

# Date -> datetime
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Symbol -> ticker (uppercase + strip)
df["ticker"] = df["symbol"].astype(str).str.upper().str.strip()
df = df.drop(columns=["symbol"])

# Numeric columns
numeric_cols = ["open", "high", "low", "close", "volume"]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Sort by ticker and date
df = df.sort_values(["ticker", "date"]).reset_index(drop=True)

required_cols = ["date", "ticker", "open", "high", "low", "close", "volume"]
missing_required_cols = [col for col in required_cols if col not in df.columns]
if missing_required_cols:
    raise ValueError(f"Missing required columns after normalization: {missing_required_cols}")

print("Normalization completed")
print("Required columns:", required_cols)
print()
print("Data types:")
print(df.dtypes)
print()
print(f"Shape: {df.shape}")
print(f"Tickers: {df['ticker'].nunique()}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")


Normalization completed
Required columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume']

Data types:
date      datetime64[ns]
open             float64
high             float64
low              float64
close            float64
volume             int64
ticker            object
dtype: object

Shape: (43932, 7)
Tickers: 30
Date range: 2020-01-02 00:00:00 to 2025-11-28 00:00:00


## Bước 1: Chốt universe nghiên cứu

Universe mục tiêu của downstream pipeline là `29` mã VN30 sau khi loại `SSB`.

Lý do loại `SSB`:
- `SSB` xuất hiện muộn do IPO muộn
- history ngắn hơn đáng kể so với phần còn lại của universe
- giữ `SSB` sẽ làm coverage theo ngày kém hơn và phá mục tiêu tạo balanced panel cho các notebook downstream


In [4]:
# VN30 reference list
vn30_list = [
    "ACB",
    "BCM",
    "BID",
    "CTG",
    "DGC",
    "FPT",
    "GAS",
    "GVR",
    "HDB",
    "HPG",
    "LPB",
    "MBB",
    "MSN",
    "MWG",
    "PLX",
    "SAB",
    "SHB",
    "SSB",
    "SSI",
    "STB",
    "TCB",
    "TPB",
    "VCB",
    "VHM",
    "VIB",
    "VIC",
    "VJC",
    "VNM",
    "VPB",
    "VRE",
]

# Remove SSB from research universe
vn29_list = [t for t in vn30_list if t != "SSB"]

print(f"VN30 original: {len(vn30_list)} tickers")
print(f"VN29 (without SSB): {len(vn29_list)} tickers")
print()
print("Before filtering:")
print(f"  Total rows: {len(df):,}")
print(f"  Tickers: {sorted(df['ticker'].unique())}")

# Keep only VN29
df = df[df["ticker"].isin(vn29_list)].copy()

post_universe_rows = len(df)
post_universe_tickers = int(df["ticker"].nunique())
post_universe_date_min = df["date"].min()
post_universe_date_max = df["date"].max()
post_universe_trading_days = int(df["date"].nunique())

print()
print("After filtering (removed SSB):")
print(f"  Total rows: {post_universe_rows:,}")
print(f"  Tickers: {post_universe_tickers}")
print(f"  Date range: {post_universe_date_min} to {post_universe_date_max}")
print(f"  Trading days: {post_universe_trading_days}")
print(f"  Tickers list: {sorted(df['ticker'].unique())}")


VN30 original: 30 tickers
VN29 (without SSB): 29 tickers

Before filtering:
  Total rows: 43,932
  Tickers: ['ACB', 'BCM', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSB', 'SSI', 'STB', 'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VRE']

After filtering (removed SSB):
  Total rows: 42,760
  Tickers: 29
  Date range: 2020-01-02 00:00:00 to 2025-11-28 00:00:00
  Trading days: 1476
  Tickers list: ['ACB', 'BCM', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSI', 'STB', 'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VRE']


## Bước 2: Xử lý Duplicate


In [5]:
# Duplicate guard
rows_before_duplicates = len(df)
n_full_dup_before = int(df.duplicated().sum())
n_key_dup_before = int(df.duplicated(subset=["date", "ticker"]).sum())

print("Before removing duplicates:")
print(f"  Full row duplicates: {n_full_dup_before}")
print(f"  (date, ticker) duplicates: {n_key_dup_before}")
print(f"  Total rows: {rows_before_duplicates:,}")

# Remove full duplicates
df = df.drop_duplicates()

# Remove (date, ticker) duplicates (keep last)
df = df.drop_duplicates(subset=["date", "ticker"], keep="last")

rows_after_duplicates = len(df)
duplicate_rows_removed = rows_before_duplicates - rows_after_duplicates

print()
print("After removing duplicates:")
print(f"  Total rows: {rows_after_duplicates:,}")
print(f"  Rows removed: {duplicate_rows_removed}")


Before removing duplicates:
  Full row duplicates: 0
  (date, ticker) duplicates: 0
  Total rows: 42,760

After removing duplicates:
  Total rows: 42,760
  Rows removed: 0


## Bước 3: Loại dòng vi phạm OHLC & giá trị vô lý

Ở pass raw EDA hiện tại:
- có `2` dòng vi phạm logic OHLC và đây là lỗi dữ liệu cần loại
- có `1` dòng `volume == 0`

Decision rule ở notebook này:
- giá `<= 0` và `volume < 0` được xem là invalid và bị loại
- `volume == 0` không bị coi là invalid theo mặc định; nếu số lượng rất ít thì ưu tiên giữ nếu dòng đó vẫn hợp lệ theo các rule còn lại
- nếu dòng `volume == 0` đồng thời vi phạm logic OHLC hoặc rule dữ liệu khác, dòng đó vẫn bị loại như một invalid row bình thường


In [6]:
# OHLC/basic-value validation
n_zero_volume_observed = int((df["volume"] == 0).sum())
print("Policy note: keep volume == 0 rows only if they remain valid under OHLC/basic-value rules.")
print(f"Observed volume == 0 rows before validation: {n_zero_volume_observed}")

valid_mask = (
    (df["open"] > 0)
    & (df["high"] > 0)
    & (df["low"] > 0)
    & (df["close"] > 0)
    & (df["volume"] >= 0)
    & (df["low"] <= df[["open", "close"]].min(axis=1))
    & (df["high"] >= df[["open", "close"]].max(axis=1))
    & (df["high"] >= df["low"])
)

invalid_rows = df[~valid_mask].copy()
n_invalid = len(invalid_rows)

print("Before OHLC validation:")
print(f"  Total rows: {len(df):,}")
print(f"  Invalid rows: {n_invalid}")

if n_invalid > 0:
    print()
    print("  Violation details:")
    print(f"    open <= 0: {(df['open'] <= 0).sum()}")
    print(f"    high <= 0: {(df['high'] <= 0).sum()}")
    print(f"    low <= 0: {(df['low'] <= 0).sum()}")
    print(f"    close <= 0: {(df['close'] <= 0).sum()}")
    print(f"    volume < 0: {(df['volume'] < 0).sum()}")
    print(f"    volume == 0: {(df['volume'] == 0).sum()}")
    print(
        f"    low > min(open, close): {(df['low'] > df[['open', 'close']].min(axis=1)).sum()}"
    )
    print(
        f"    high < max(open, close): {(df['high'] < df[['open', 'close']].max(axis=1)).sum()}"
    )
    print(f"    high < low: {(df['high'] < df['low']).sum()}")

    invalid_rows.to_csv(REPORTS_DIR / "invalid_ohlc_rows.csv", index=False)
    print()
    print(f"  Saved invalid rows to: {REPORTS_DIR / 'invalid_ohlc_rows.csv'}")

# Keep only valid rows
df = df[valid_mask].copy()
n_zero_volume_kept = int((df["volume"] == 0).sum())
n_zero_volume_removed = n_zero_volume_observed - n_zero_volume_kept

print()
print("After OHLC validation:")
print(f"  Total rows: {len(df):,}")
print(f"  Rows removed: {n_invalid}")
print(f"  volume == 0 kept: {n_zero_volume_kept}")
print(f"  volume == 0 removed due to other invalid rules: {n_zero_volume_removed}")


Policy note: keep volume == 0 rows only if they remain valid under OHLC/basic-value rules.
Observed volume == 0 rows before validation: 1
Before OHLC validation:
  Total rows: 42,760
  Invalid rows: 2

  Violation details:
    open <= 0: 0
    high <= 0: 0
    low <= 0: 0
    close <= 0: 0
    volume < 0: 0
    volume == 0: 1
    low > min(open, close): 2
    high < max(open, close): 0
    high < low: 0

  Saved invalid rows to: d:\Archive\NCKH\CODE\vn30-ver1\reports\preprocess\invalid_ohlc_rows.csv

After OHLC validation:
  Total rows: 42,758
  Rows removed: 2
  volume == 0 kept: 0
  volume == 0 removed due to other invalid rules: 1


## Bước 4: Xử lý Missing (NaN)


In [7]:
# Missing-value guard
missing_rows_before = len(df)
print("Before handling missing values:")
print(f"  Total rows: {missing_rows_before:,}")
print()
print("  Missing values by column:")

has_any_missing = False
for col in ["date", "ticker", "open", "high", "low", "close", "volume"]:
    n_missing = int(df[col].isna().sum())
    if n_missing > 0:
        has_any_missing = True
        print(f"    {col}: {n_missing} ({n_missing/len(df)*100:.2f}%)")

if not has_any_missing:
    print("    None")

# Drop rows with missing date or ticker
df = df.dropna(subset=["date", "ticker"])

# Drop rows with missing price columns (open, high, low, close)
price_cols = ["open", "high", "low", "close"]
df = df.dropna(subset=price_cols)

# For volume: drop if NaN (do not fill mechanically)
df = df.dropna(subset=["volume"])

missing_rows_removed = missing_rows_before - len(df)

print()
print("After removing missing values:")
print(f"  Total rows: {len(df):,}")
print(f"  Rows removed: {missing_rows_removed}")
print()
print("  Remaining missing values:")
missing_check = (
    df[["date", "ticker", "open", "high", "low", "close", "volume"]].isna().sum()
)
print(missing_check[missing_check > 0] if missing_check.sum() > 0 else "  None")


Before handling missing values:
  Total rows: 42,758

  Missing values by column:
    None

After removing missing values:
  Total rows: 42,758
  Rows removed: 0

  Remaining missing values:
  None


## Bước 5: Kiểm tra coverage theo mã

Bước này dùng để xác nhận sau khi loại `SSB`, phần universe còn lại có đủ coverage để giữ nguyên cho downstream. Nếu một mã còn lại có coverage thấp bất thường thì mới xem xét loại thêm, còn nếu coverage vẫn cao đồng đều thì chỉ cần ghi nhận và giữ nguyên universe.


In [8]:
# Coverage by ticker
total_trading_days = df["date"].nunique()
coverage_by_ticker = (
    df.groupby("ticker").agg({"date": ["count", "min", "max"]}).round(2)
)

coverage_by_ticker.columns = ["n_days", "first_date", "last_date"]
coverage_by_ticker["coverage_pct"] = (
    coverage_by_ticker["n_days"] / total_trading_days * 100
).round(2)
coverage_by_ticker = coverage_by_ticker.sort_values("n_days")

print("=== COVERAGE BY TICKER ===")
print(f"Total unique trading days: {total_trading_days}")
print()
print("Coverage statistics:")
print(coverage_by_ticker)

threshold = 0.90  # 90% coverage
low_coverage = coverage_by_ticker[coverage_by_ticker["coverage_pct"] < threshold * 100]
low_coverage_tickers = low_coverage.index.tolist()

if len(low_coverage) > 0:
    print()
    print(f"Tickers with coverage < {threshold*100}%:")
    print(low_coverage)
    coverage_by_ticker_decision = "Review low-coverage tickers before downstream modeling."
    print()
    print(f"Recommendation: {coverage_by_ticker_decision}")
else:
    coverage_by_ticker_decision = "No additional ticker removed after SSB; remaining universe coverage is acceptable."
    print()
    print(f"All tickers have coverage >= {threshold*100}%")
    print(f"Decision: {coverage_by_ticker_decision}")


=== COVERAGE BY TICKER ===
Total unique trading days: 1476

Coverage statistics:
        n_days first_date  last_date  coverage_pct
ticker                                            
LPB       1466 2020-01-02 2025-11-28         99.32
BCM       1467 2020-01-02 2025-11-28         99.39
VIB       1469 2020-01-02 2025-11-28         99.53
DGC       1470 2020-01-02 2025-11-28         99.59
GVR       1470 2020-01-02 2025-11-28         99.59
ACB       1471 2020-01-02 2025-11-28         99.66
SHB       1473 2020-01-02 2025-11-28         99.80
VNM       1476 2020-01-02 2025-11-28        100.00
VJC       1476 2020-01-02 2025-11-28        100.00
VIC       1476 2020-01-02 2025-11-28        100.00
VHM       1476 2020-01-02 2025-11-28        100.00
VCB       1476 2020-01-02 2025-11-28        100.00
TPB       1476 2020-01-02 2025-11-28        100.00
TCB       1476 2020-01-02 2025-11-28        100.00
STB       1476 2020-01-02 2025-11-28        100.00
SSI       1476 2020-01-02 2025-11-28        100.00
P

**Kết luận ngắn.** Sau khi loại `SSB`, toàn bộ `29` mã còn lại đều có coverage đủ cao theo ngưỡng kiểm tra hiện tại, nên notebook không loại thêm ticker nào ở bước này. Quyết định cân bằng panel vì vậy được thực hiện ở mức **ngày giao dịch**, không phải loại thêm **mã cổ phiếu**.


## Bước 6: Xử lý coverage theo ngày để tạo balanced panel

Rule đang dùng trong notebook này là đếm số ticker trong universe nghiên cứu `29` mã trên từng ngày. Ngày nào có `< 29` ticker sẽ được coi là incomplete day.

Lưu ý: số ngày coverage thấp ở đây có thể khác notebook `01_eda.ipynb` vì:
- preprocess đã chuyển sang universe `29` mã sau khi loại `SSB`
- định nghĩa coverage ở đây dùng trực tiếp rule balanced panel `< 29 ticker/day`
- mục tiêu ở đây là tạo panel sạch cho downstream matrix-based notebooks, không phải chỉ audit raw coverage


In [9]:
# Coverage by day
tickers_per_day = df.groupby("date")["ticker"].nunique()
rows_before_incomplete_filter = len(df)

print("=== TICKERS PER DAY ANALYSIS ===")
print()
print("Statistics:")
print(tickers_per_day.describe())

expected_tickers = 29  # VN29 after removing SSB
incomplete_days = tickers_per_day[tickers_per_day < expected_tickers]
incomplete_days_found = len(incomplete_days)
incomplete_days_pct = incomplete_days_found / len(tickers_per_day) * 100
incomplete_days_removed = 0
incomplete_rows_removed = 0
coverage_day_decision = "No incomplete day found."

print()
print(f"=== INCOMPLETE DAYS (< {expected_tickers} tickers) ===")
print(f"Number of incomplete days: {incomplete_days_found}")
print(f"Percentage: {incomplete_days_pct:.2f}%")

if incomplete_days_found > 0:
    print()
    print("Details:")
    print(incomplete_days.sort_values())

    incomplete_days_df = pd.DataFrame(
        {"date": incomplete_days.index, "n_tickers": incomplete_days.values}
    )
    incomplete_days_df.to_csv(REPORTS_DIR / "incomplete_days.csv", index=False)
    print()
    print(f"Saved to: {REPORTS_DIR / 'incomplete_days.csv'}")

    if incomplete_days_found < 0.05 * len(tickers_per_day):
        coverage_day_decision = (
            f"Drop {incomplete_days_found} incomplete days to create a balanced 29-ticker panel."
        )
        print()
        print(f"Decision: DROP these {incomplete_days_found} days (< 5% of total)")
        df = df[~df["date"].isin(incomplete_days.index)].copy()
        incomplete_days_removed = incomplete_days_found
        incomplete_rows_removed = rows_before_incomplete_filter - len(df)
        print(f"Rows after dropping incomplete days: {len(df):,}")
        print(f"Rows removed by day-level filtering: {incomplete_rows_removed:,}")
    else:
        coverage_day_decision = (
            f"Found {incomplete_days_found} incomplete days; too many to auto-drop under the current rule."
        )
        print()
        print(f"Too many incomplete days ({incomplete_days_found}), review manually")
else:
    print()
    print(f"All days have {expected_tickers} tickers")

print()
print("Note: this count can differ from raw EDA because preprocess uses VN29 universe and a balanced-panel coverage rule.")


=== TICKERS PER DAY ANALYSIS ===

Statistics:
count    1476.000000
mean       28.968835
std         0.195832
min        27.000000
25%        29.000000
50%        29.000000
75%        29.000000
max        29.000000
Name: ticker, dtype: float64

=== INCOMPLETE DAYS (< 29 tickers) ===
Number of incomplete days: 40
Percentage: 2.71%

Details:
date
2020-11-05    27
2020-10-30    27
2020-11-02    27
2020-11-03    27
2020-11-04    27
2020-11-06    27
2020-10-27    28
2020-10-28    28
2020-10-29    28
2020-01-21    28
2020-11-09    28
2020-12-02    28
2020-12-03    28
2020-12-04    28
2020-12-07    28
2020-12-08    28
2021-10-06    28
2020-10-26    28
2020-08-28    28
2020-08-27    28
2020-08-26    28
2020-02-27    28
2020-03-09    28
2020-03-10    28
2020-03-11    28
2020-03-12    28
2020-03-13    28
2020-03-16    28
2020-07-20    28
2020-07-21    28
2020-07-22    28
2020-07-23    28
2020-07-24    28
2020-07-27    28
2020-08-20    28
2020-08-21    28
2020-08-24    28
2020-08-25    28
2021-10-

## Bước 7: Đánh dấu Outliers (KHÔNG xóa)

Outlier được **flag** chứ không bị xóa trực tiếp. Lý do là các biến động cực trị ở `range_ratio` hoặc `volume` có thể phản ánh diễn biến thị trường thật, và downstream notebooks có thể cần giữ lại thông tin này để phân tích hoặc xử lý riêng.


In [10]:
# Calculate features for outlier detection
df["range_ratio"] = (df["high"] - df["low"]) / df["close"] * 100

# Calculate thresholds
range_threshold = df["range_ratio"].quantile(0.99)
volume_threshold = df["volume"].quantile(0.99)

print(f"=== OUTLIER THRESHOLDS (99th percentile) ===")
print(f"Range ratio: {range_threshold:.2f}%")
print(f"Volume: {volume_threshold:,.0f}")

# Flag outliers (but don't remove)
df["is_range_outlier"] = df["range_ratio"] > range_threshold
df["is_volume_outlier"] = df["volume"] > volume_threshold

n_range_outliers = df["is_range_outlier"].sum()
n_volume_outliers = df["is_volume_outlier"].sum()
n_both = (df["is_range_outlier"] & df["is_volume_outlier"]).sum()

print(f"\n=== OUTLIER FLAGGING ===")
print(f"Range outliers: {n_range_outliers} ({n_range_outliers/len(df)*100:.2f}%)")
print(f"Volume outliers: {n_volume_outliers} ({n_volume_outliers/len(df)*100:.2f}%)")
print(f"Both: {n_both}")

print(f"\n✓ Outliers flagged but NOT removed")
print(f"  Total rows: {len(df):,}")


=== OUTLIER THRESHOLDS (99th percentile) ===
Range ratio: 8.95%
Volume: 52,180,422

=== OUTLIER FLAGGING ===
Range outliers: 417 (1.00%)
Volume outliers: 417 (1.00%)
Both: 16

✓ Outliers flagged but NOT removed
  Total rows: 41,644


## Bước 8: Xuất file clean_ohlcv.csv


In [11]:
# Prepare final dataframe
df_clean = df[
    [
        "date",
        "ticker",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "range_ratio",
        "is_range_outlier",
        "is_volume_outlier",
    ]
].copy()

# Sort by ticker and date
df_clean = df_clean.sort_values(["ticker", "date"]).reset_index(drop=True)

# Save to CSV
output_path = DATA_PROCESSED / "clean_ohlcv.csv"
df_clean.to_csv(output_path, index=False)

print(f"=== CLEAN DATA EXPORT ===")
print(f"✓ Saved to: {output_path}")
print(f"\nFinal dataset:")
print(f"  Rows: {len(df_clean):,}")
print(f"  Columns: {list(df_clean.columns)}")
print(f"  Tickers: {df_clean['ticker'].nunique()}")
print(f"  Date range: {df_clean['date'].min()} to {df_clean['date'].max()}")
print(f"  Trading days: {df_clean['date'].nunique()}")

print(f"\nFirst rows:")
df_clean.head(10)


=== CLEAN DATA EXPORT ===
✓ Saved to: d:\Archive\NCKH\CODE\vn30-ver1\data\processed\clean_ohlcv.csv

Final dataset:
  Rows: 41,644
  Columns: ['date', 'ticker', 'open', 'high', 'low', 'close', 'volume', 'range_ratio', 'is_range_outlier', 'is_volume_outlier']
  Tickers: 29
  Date range: 2020-01-02 00:00:00 to 2025-11-28 00:00:00
  Trading days: 1436

First rows:


,date,ticker,open,high,low,close,volume,range_ratio,is_range_outlier,is_volume_outlier
0,2020-01-02,ACB,6.58,6.64,6.55,6.64,1163109,1.355422,False,False
1,2020-01-03,ACB,6.64,6.70,6.61,6.64,1055528,1.355422,False,False
2,2020-01-06,ACB,6.64,6.64,6.49,6.49,1286035,2.311248,False,False
3,2020-01-07,ACB,6.49,6.55,6.49,6.49,1050934,0.924499,False,False
4,2020-01-08,ACB,6.49,6.49,6.35,6.38,2304937,2.194357,False,False
5,2020-01-09,ACB,6.41,6.52,6.41,6.46,1422248,1.702786,False,False
6,2020-01-10,ACB,6.49,6.67,6.49,6.55,2371501,2.748092,False,False
7,2020-01-13,ACB,6.61,6.61,6.55,6.61,1523181,0.907716,False,False
8,2020-01-14,ACB,6.61,6.78,6.61,6.78,2851982,2.507375,False,False
9,2020-01-15,ACB,6.78,6.78,6.70,6.72,1158657,1.190476,False,False


## Bước 9: Sanity Check cuối cùng

Bước này xác nhận clean panel đã thỏa các điều kiện bắt buộc cho downstream. Notebook vẫn in `PASS/FAIL` để dễ đọc, và bổ sung `assert` cho các invariants cứng nhằm chặn trạng thái dữ liệu không hợp lệ trước khi sang các notebook tiếp theo.


In [12]:
print("=" * 60)
print("SANITY CHECK - FINAL VALIDATION")
print("=" * 60)

# 1. Check SSB not in tickers
print()
print("1. SSB removed from universe:")
has_ssb = "SSB" in df_clean["ticker"].unique()
print(f"   {'FAIL' if has_ssb else 'PASS'} - SSB in data: {has_ssb}")
print(f"   Tickers: {sorted(df_clean['ticker'].unique())}")

# 2. Check OHLC rules
print()
print("2. OHLC rules validation:")
ohlc_checks = {
    "open > 0": (df_clean["open"] > 0).all(),
    "high > 0": (df_clean["high"] > 0).all(),
    "low > 0": (df_clean["low"] > 0).all(),
    "close > 0": (df_clean["close"] > 0).all(),
    "volume >= 0": (df_clean["volume"] >= 0).all(),
    "low <= min(o,c)": (
        df_clean["low"] <= df_clean[["open", "close"]].min(axis=1)
    ).all(),
    "high >= max(o,c)": (
        df_clean["high"] >= df_clean[["open", "close"]].max(axis=1)
    ).all(),
    "high >= low": (df_clean["high"] >= df_clean["low"]).all(),
}

for rule, passed in ohlc_checks.items():
    status = "PASS" if passed else "FAIL"
    print(f"   {status} - {rule}")

# 3. Check no missing in core columns
print()
print("3. Missing values check:")
core_cols = ["date", "ticker", "open", "high", "low", "close", "volume"]
has_missing_core = df_clean[core_cols].isna().any().any()
print(
    f"   {'FAIL' if has_missing_core else 'PASS'} - Missing in core columns: {has_missing_core}"
)

# 4. Check tickers per day
print()
print("4. Tickers per day (after cleanup):")
tickers_per_day_final = df_clean.groupby("date")["ticker"].nunique()
coverage_pass = tickers_per_day_final.min() == 29
print(f"   Min: {tickers_per_day_final.min()}")
print(f"   Median: {tickers_per_day_final.median()}")
print(f"   Max: {tickers_per_day_final.max()}")
print(
    f"   {'PASS' if coverage_pass else 'FAIL'} - Days with 29 tickers: {(tickers_per_day_final == 29).sum()} / {len(tickers_per_day_final)}"
)

# 5. Check duplicates
print()
print("5. Duplicates check:")
n_duplicates = int(df_clean.duplicated(subset=["date", "ticker"]).sum())
print(
    f"   {'FAIL' if n_duplicates > 0 else 'PASS'} - (date, ticker) duplicates: {n_duplicates}"
)

# 6. Check final universe size
print()
print("6. Final universe size:")
final_ticker_count = int(df_clean["ticker"].nunique())
print(
    f"   {'PASS' if final_ticker_count == 29 else 'FAIL'} - Final tickers: {final_ticker_count}"
)

print()
print("=" * 60)
print("SANITY CHECK COMPLETED")
print("=" * 60)

assert not has_ssb, "SSB should not appear in clean dataset"
assert n_duplicates == 0, "Duplicate (date, ticker) rows remain in clean dataset"
assert not has_missing_core, "Missing values remain in required core columns"
assert all(ohlc_checks.values()), "OHLC validation failed for clean dataset"
assert coverage_pass, "Balanced panel check failed: some days still have < 29 tickers"
assert final_ticker_count == 29, "Final ticker count is not equal to the research universe"


SANITY CHECK - FINAL VALIDATION

1. SSB removed from universe:
   PASS - SSB in data: False
   Tickers: ['ACB', 'BCM', 'BID', 'CTG', 'DGC', 'FPT', 'GAS', 'GVR', 'HDB', 'HPG', 'LPB', 'MBB', 'MSN', 'MWG', 'PLX', 'SAB', 'SHB', 'SSI', 'STB', 'TCB', 'TPB', 'VCB', 'VHM', 'VIB', 'VIC', 'VJC', 'VNM', 'VPB', 'VRE']

2. OHLC rules validation:
   PASS - open > 0
   PASS - high > 0
   PASS - low > 0
   PASS - close > 0
   PASS - volume >= 0
   PASS - low <= min(o,c)
   PASS - high >= max(o,c)
   PASS - high >= low

3. Missing values check:
   PASS - Missing in core columns: False

4. Tickers per day (after cleanup):
   Min: 29
   Median: 29.0
   Max: 29
   PASS - Days with 29 tickers: 1436 / 1436

5. Duplicates check:
   PASS - (date, ticker) duplicates: 0

6. Final universe size:
   PASS - Final tickers: 29

SANITY CHECK COMPLETED


## Tạo Preprocessing Report

Report cuối notebook phải ghi lại không chỉ số lượng dòng/mã trước sau preprocess mà còn decision logic bắt nguồn từ raw EDA. Phần này dùng để lưu audit trail cho các bước loại, giữ, và flag trong quá trình tạo clean panel.


In [13]:
# Create preprocessing report
report = {
    "preprocessing_date": pd.Timestamp.now().strftime("%Y-%m-%d %H:%M:%S"),
    "input_file": str(DATA_RAW / "vn30_stock_data.csv"),
    "output_file": str(DATA_PROCESSED / "clean_ohlcv.csv"),
    "canonical_output_schema": [
        "date",
        "ticker",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "range_ratio",
        "is_range_outlier",
        "is_volume_outlier",
    ],
    "original_data": {
        "rows": len(df_raw),
        "tickers": int(df_raw["symbol"].nunique()) if "symbol" in df_raw.columns else 0,
        "date_min": str(pd.to_datetime(df_raw["date"], errors="coerce").min()) if "date" in df_raw.columns else None,
        "date_max": str(pd.to_datetime(df_raw["date"], errors="coerce").max()) if "date" in df_raw.columns else None,
    },
    "universe_after_filter": {
        "rows": int(post_universe_rows),
        "tickers": int(post_universe_tickers),
        "date_min": str(post_universe_date_min),
        "date_max": str(post_universe_date_max),
        "trading_days": int(post_universe_trading_days),
    },
    "final_data": {
        "rows": len(df_clean),
        "tickers": int(df_clean["ticker"].nunique()),
        "date_min": str(df_clean["date"].min()),
        "date_max": str(df_clean["date"].max()),
        "trading_days": int(df_clean["date"].nunique()),
    },
    "summary_counts": {
        "raw_rows": int(len(df_raw)),
        "raw_tickers": int(df_raw["symbol"].nunique()) if "symbol" in df_raw.columns else 0,
        "post_universe_rows": int(post_universe_rows),
        "post_universe_tickers": int(post_universe_tickers),
        "duplicates_removed": int(duplicate_rows_removed),
        "full_row_duplicates_before": int(n_full_dup_before),
        "date_ticker_duplicates_before": int(n_key_dup_before),
        "ohlc_invalid_removed": int(n_invalid),
        "missing_removed": int(missing_rows_removed),
        "incomplete_days_found": int(incomplete_days_found),
        "incomplete_days_removed": int(incomplete_days_removed),
        "incomplete_rows_removed": int(incomplete_rows_removed),
        "final_rows": int(len(df_clean)),
        "final_tickers": int(df_clean["ticker"].nunique()),
        "final_trading_days": int(df_clean["date"].nunique()),
        "zero_volume_rows_observed": int(n_zero_volume_observed),
        "zero_volume_rows_kept": int(n_zero_volume_kept),
    },
    "cleaning_steps": {
        "step1_removed_ssb": "Removed SSB from universe (VN30 -> VN29)",
        "step2_duplicates_removed": f"Removed {duplicate_rows_removed} rows via full-row and (date, ticker) dedup guard",
        "step3_invalid_ohlc_removed": f"Removed {n_invalid} rows with OHLC/basic-value violations",
        "step4_missing_removed": f"Removed {missing_rows_removed} rows with missing required values",
        "step5_incomplete_days_removed": f"Removed {incomplete_days_removed} days and {incomplete_rows_removed} rows with < 29 tickers to create a balanced panel",
        "step6_outliers_flagged": f"Flagged {n_range_outliers} range outliers and {n_volume_outliers} volume outliers without removing them",
    },
    "coverage_by_ticker": {
        "threshold_pct": 90.0,
        "low_coverage_tickers": low_coverage_tickers,
        "decision": coverage_by_ticker_decision,
    },
    "coverage_by_day": {
        "expected_tickers": 29,
        "incomplete_days_found": int(incomplete_days_found),
        "incomplete_days_pct": round(float(incomplete_days_pct), 4),
        "incomplete_days_removed": int(incomplete_days_removed),
        "incomplete_rows_removed": int(incomplete_rows_removed),
        "decision": coverage_day_decision,
        "note_vs_raw_eda": "Counts can differ from raw EDA because preprocess uses VN29 universe and the balanced-panel rule < 29 tickers/day.",
    },
    "zero_volume_policy": {
        "observed_rows": int(n_zero_volume_observed),
        "kept_rows": int(n_zero_volume_kept),
        "removed_due_to_other_rules": int(n_zero_volume_removed),
        "policy": "Keep volume == 0 rows only if they remain valid under OHLC/basic-value checks.",
    },
    "data_quality": {
        "tickers_per_day_min": int(tickers_per_day_final.min()),
        "tickers_per_day_median": int(tickers_per_day_final.median()),
        "tickers_per_day_max": int(tickers_per_day_final.max()),
        "days_with_full_coverage": int((tickers_per_day_final == 29).sum()),
    },
    "eda_informed_decisions": {
        "raw_data_status": "Raw data is relatively clean: no duplicate, no missing, no parse-error issue in the current snapshot.",
        "main_issues": [
            "Coverage inconsistency is the main panel-level issue.",
            "A very small number of volume == 0 rows should be kept only if otherwise valid.",
            "OHLC-invalid rows are treated as data errors and removed.",
            "SSB is excluded because IPO timing makes its history materially shorter than the research universe.",
        ],
        "research_universe_note": "The clean panel is designed for return, correlation, clustering, and optimization notebooks downstream.",
    },
    "notes": [
        "SSB removed due to materially shorter history / IPO timing.",
        "Outliers flagged but not removed for downstream handling.",
        "All OHLC validation rules passed in the final clean dataset.",
        "No missing values remain in required core columns.",
        "Balanced 29-ticker panel created for downstream modeling.",
    ],
}

report_path = REPORTS_DIR / "preprocessing_report.json"
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, indent=2, ensure_ascii=False)

print(f"Preprocessing report saved to: {report_path}")
print()
print("=== PREPROCESSING SUMMARY ===")
print(f"Original: {report['original_data']['rows']:,} rows")
print(f"After universe filter: {report['universe_after_filter']['rows']:,} rows")
print(f"Final: {report['final_data']['rows']:,} rows")
print(
    f"Removed: {report['original_data']['rows'] - report['final_data']['rows']:,} rows"
)
print(f"Tickers: {report['final_data']['tickers']}")
print(
    f"Date range: {report['final_data']['date_min']} to {report['final_data']['date_max']}"
)
print(f"Trading days: {report['final_data']['trading_days']}")
print(f"Zero-volume rows kept: {report['summary_counts']['zero_volume_rows_kept']}")


Preprocessing report saved to: d:\Archive\NCKH\CODE\vn30-ver1\reports\preprocess\preprocessing_report.json

=== PREPROCESSING SUMMARY ===
Original: 43,932 rows
After universe filter: 42,760 rows
Final: 41,644 rows
Removed: 2,288 rows
Tickers: 29
Date range: 2020-01-02 00:00:00 to 2025-11-28 00:00:00
Trading days: 1436
Zero-volume rows kept: 0


## K?t lu?n & N?i sang downstream

Sau preprocess, d? li?u ?? tr? th?nh **clean panel c?n b?ng** cho universe nghi?n c?u g?m `29` ticker theo ng?y, kh?ng c?n duplicate `(date, ticker)`, kh?ng c?n missing ? c?c c?t c?t l?i, v? th?a c?c rule OHLC c?n thi?t cho c?c b??c downstream.

C?c quy?t ??nh ch?nh ?? ?p d?ng trong notebook n?y:
- lo?i `SSB` kh?i universe nghi?n c?u
- lo?i c?c d?ng `OHLC-invalid` / `basic-value invalid`
- gi? outlier nh?ng ch? `flag` qua `is_range_outlier` v? `is_volume_outlier`
- lo?i `incomplete trading days` ?? t?o **balanced panel**
- gi? policy r? r?ng cho `volume == 0`: ch? gi? n?u d?ng v?n h?p l? theo c?c rule c?n l?i

Artifact canonical ??u ra c?a notebook n?y l? `data/processed/clean_ohlcv.csv`.

C?c notebook downstream s? d?ng clean panel n?y ?? t?nh return, t?o ma tr?n return s?ch, r?i ti?p t?c c?c b??c correlation, clustering v? optimization tr?n c?ng m?t input ?? ???c l?m s?ch nh?t qu?n.
